Imports and Setup for the project

In [10]:
import os
import numpy as np

DATA_DIRECTORY = "data\out"
LAB_DIRECTORY = "data\cmu_us_slt_arctic\lab"
TRAIN_FILE = os.path.join(DATA_DIRECTORY, "train.txt")
VAL_FILE = os.path.join(DATA_DIRECTORY, "val.txt")
TEST_FILE = os.path.join(DATA_DIRECTORY, "test.txt")

def get_basenames(file_path):
    with open(file_path, 'r') as f:
        basenames = [line.strip() for line in f.readlines()]
    return basenames

train_basenames = get_basenames(TRAIN_FILE)
val_basenames = get_basenames(VAL_FILE)
test_basenames = get_basenames(TEST_FILE)

print(f"Loaded {len(train_basenames)} training utterances.")
print(f"Loaded {len(val_basenames)} validation utterances.")
print(f"Loaded {len(test_basenames)} test utterances.")

Loaded 792 training utterances.
Loaded 113 validation utterances.
Loaded 227 test utterances.


<>:4: SyntaxWarning: invalid escape sequence '\o'
<>:5: SyntaxWarning: invalid escape sequence '\c'
<>:4: SyntaxWarning: invalid escape sequence '\o'
<>:5: SyntaxWarning: invalid escape sequence '\c'
C:\Users\aubie\AppData\Local\Temp\ipykernel_15728\2051616938.py:4: SyntaxWarning: invalid escape sequence '\o'
  DATA_DIRECTORY = "data\out"
C:\Users\aubie\AppData\Local\Temp\ipykernel_15728\2051616938.py:5: SyntaxWarning: invalid escape sequence '\c'
  LAB_DIRECTORY = "data\cmu_us_slt_arctic\lab"


Phoneme Mapping and Frame Alignment

In [11]:
PHONEMES = ['pau', 'iy', 'ih', 'eh', 'ae', 'aa', 'ah', 'ao', 'uh', 'uw', 'er', 
    'ax', 'ix', 'ey', 'ay', 'oy', 'aw', 'ow', 'l', 'r', 'y', 'w', 
    'er', 'm', 'n', 'ng', 'ch', 'jh', 'dh', 'b', 'd', 'dx', 'g', 
    'p', 't', 'k', 'z', 'zh', 'v', 'f', 'th', 's', 'sh', 'hh'
    ]

phoneme_to_id = {p: i for i, p in enumerate(PHONEMES)}
id_to_phoneme = {i: p for p, i in phoneme_to_id.items()}
K = len(PHONEMES)
print(f"Number of unique phonemes: {K}")

def align_frames(lab_path, num_frames, frame_shift=0.01):
    labels = np.zeros(num_frames, dtype=int)
    with open(lab_path, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 3:
                start, end, ph = parts
                start_frame = int(float(start) / frame_shift)
                end_frame = int(float(end) / frame_shift)
                
                # Handle edge cases where the label might not be in our standard list
                if ph in phoneme_to_id:
                    labels[start_frame:min(end_frame, num_frames)] = phoneme_to_id[ph]
    return labels



Number of unique phonemes: 44


Supervised Parameter Estimation

In [13]:
# Initialize parameters for the model
pi = np.zeros(K)  # Initial state probabilities
A = np.zeros((K, K))  # Transition probabilities
obs_by_state = {i: [] for i in range(K)}  # Observations grouped by state

print("Processing training files to estimate HMM parameters")

for basename in train_basenames:
    feature_path = os.path.join(DATA_DIRECTORY, f"{basename}.npy")
    label_path = os.path.join(LAB_DIRECTORY, f"{basename}.lab")

    obs = np.load(feature_path)  # Shape: (num_frames, feature_dim)
    T = len(obs)

    states = align_frames(label_path, T)  

    pi[states[0]] += 1  # Increment initial state count
    for t in range(T - 1):
        A[states[t], states[t + 1]] += 1
    
    for t in range(T):
        obs_by_state[states[t]].append(obs[t])
    
# Normalize to get probabilities
pi_sum = np.sum(pi)
if pi_sum > 0:
    pi /= pi_sum

# Avoid division by zero for transition probabilities
row_sums = A.sum(axis=1, keepdims=True)
A = np.divide(A, row_sums, out=np.zeros_like(A), where=row_sums != 0)

# Compute Gaussian parameters for each state
D = 39
means = np.zeros((K, D))
covs = np.zeros((K, D, D))
reg_term = 1e-5 * np.eye(D)

for i in range(K):
    data = np.array(obs_by_state[i])
    if len(data) > 0:
        means[i] = np.mean(data, axis=0) 
        # rowvar=False because rows are observations, columns are variables [cite: 116]
        covs[i] = np.cov(data, rowvar=False) + reg_term 
    else:
        # Fallback if a phoneme was never observed
        covs[i] = reg_term

print("Finished estimating HMM parameters")

Processing training files to estimate HMM parameters
Finished estimating HMM parameters


Log-PDF Function

In [ ]:
def log_multivariate_normal_pdf(x, mean, cov):
    D = len(mean)
    sign, logdet = np.linalg.slogdet(cov)
    diff = x - mean
    inv_cov_dot_diff = np.linalg.solve(cov, diff)
    mahalanobis = np.dot(diff, inv_cov_dot_diff)
    return -0.5 * (D * np.log(2 * np.pi) + logdet + mahalanobis)

Inference Algorithms

Viterbi Algorithm

In [ ]:
def viterbi(obs, pi, A, means, covs):
    T = len(obs)
    K = len(pi)

    log_pi = np.log(pi + 1e-10)  # Add small value to avoid log(0)
    log_A = np.log(A + 1e-10)  # Add small value to avoid log(0)